# Generator context policies

`behaviors.rollouts.chat.run_chat_rollout` runs one conversation between two objects: a
`ChatSessionModel` (the model under study) and a `ContextPolicy` (everything else — user
messages, tools, and the decision to stop). This notebook shows the async-generator way of
writing a context policy: each `yield` produces the next context turn, the model's reply
arrives back as the value of the `yield` expression, and the rollout ends by yielding an
`EndRollout` whose `metadata` carries whatever the policy measured, as small tagged
dataclasses the policy defines. The policy reads as a
straight-line script with local variables instead of a state machine.

The example policy plays a number-guessing game against the model and records how many turns
the model needed — a stand-in for the general pattern of "drive a conversation by reacting to
model turns, then record an outcome."

Requires `ANTHROPIC_API_KEY` in the environment.

In [ ]:
import dataclasses
import re

from behaviors.implementations.context_policies import (
    ContextPolicyGenerator,
    GeneratorContextPolicy,
    context_policy_from_generator,
)
from behaviors.implementations.models import chat_session_model_from_api
from behaviors.implementations.models.anthropic import AnthropicMessagesAPI
from behaviors.rollouts.chat import run_chat_rollout
from behaviors.types import (
    ContentText,
    ContextTurn,
    DynamicData,
    EndRollout,
    ModelResponse,
    ModelTurn,
    SamplingParams,
    SessionInitialization,
    SetSystemMessage,
    SystemMessage,
    UserMessage,
    find_metadata,
)
from fxtr.entity_defns.dataclass_bases import TaggedDataclass

## 1. The model under study

Any `ModelAPI` becomes a `ChatSessionModel` through `chat_session_model_from_api`, which holds
the accumulated conversation as session state. Every turn it produces carries the raw
`ModelResponse` — stop reason, usage — in the turn's `metadata`, readable with
`find_metadata(turn.metadata, ModelResponse)`.

In [ ]:
api = AnthropicMessagesAPI("claude-haiku-4-5-20251001")
model = chat_session_model_from_api(api, SamplingParams(max_tokens=200))

## 2. A policy as a generator

Subclass `GeneratorContextPolicy`: configuration lives on the class (dataclass fields work
well), a policy whose sessions need a system prompt overrides `generate_initialization`, and
`generate_turns` is an async generator method — called once per rollout, so per-rollout state
is just its local variables. End the rollout by yielding an `EndRollout` (async generators
cannot return values, so the end is yielded like any other step).

One thing generator policies give up: a suspended generator frame cannot be captured, so they
are never `SerializableContextPolicy`s — a policy that needs checkpointing implements the
session protocol directly.

In [ ]:
def _user(text: str) -> UserMessage:
    return UserMessage(content=(ContentText(text=text),))


def _text_of(model_turn: ModelTurn) -> str:
    return " ".join(
        block.text
        for message in model_turn.messages
        for block in message.content
        if isinstance(block, ContentText)
    )


# Metadata entries are tagged dataclasses; the tag (a reverse-domain name you control) is what
# lets a reader recognize them after the rollout is stored and loaded.
@dataclasses.dataclass(frozen=True)
class ParsedGuess(TaggedDataclass, fxtr_type="org.example.guessing.ParsedGuess.v1"):
    guess: int | None


@dataclasses.dataclass(frozen=True)
class GameResult(TaggedDataclass, fxtr_type="org.example.guessing.GameResult.v1"):
    guessed: bool
    turns: int


@dataclasses.dataclass(frozen=True)
class GuessingGamePolicy(GeneratorContextPolicy):
    """Plays "guess my number": answers higher/lower until the model finds it."""

    secret: int
    limit: int = 100

    async def generate_initialization(self) -> SessionInitialization | None:
        return SetSystemMessage(
            message=SystemMessage(
                content=(
                    ContentText(
                        text="You are playing a guessing game. Each turn, guess a single"
                        " integer; the user will answer 'higher' or 'lower'. Reply with"
                        " just your guess."
                    ),
                )
            )
        )

    async def generate_turns(self) -> ContextPolicyGenerator:
        turns = 0
        model_turn = yield ContextTurn(
            messages=(_user(f"I'm thinking of a number between 1 and {self.limit}. Guess it!"),),
            available_tools=None,
        )
        while True:
            turns += 1
            match = re.search(r"-?\d+", _text_of(model_turn)) if model_turn else None
            guess = int(match.group()) if match else None
            if guess == self.secret:
                yield EndRollout(
                    metadata=(DynamicData.from_tagged(GameResult(guessed=True, turns=turns)),)
                )
                return
            hint = (
                "I couldn't find a number in that — reply with just an integer."
                if guess is None
                else ("higher" if guess < self.secret else "lower")
            )
            model_turn = yield ContextTurn(
                messages=(_user(hint),),
                available_tools=None,
                metadata=(DynamicData.from_tagged(ParsedGuess(guess)),),
            )

## 3. Run it

`run_chat_rollout` drives the exchange. The policy above never gives up on its own, so the
runner's `max_turns` is the cap — if it fires, the rollout comes back with a `RolloutMaxTurns`
status and no `end`.

In [ ]:
rollout = await run_chat_rollout(model, GuessingGamePolicy(secret=41), max_turns=12)

print(rollout.status)
print(find_metadata(rollout.end.metadata, GameResult) if rollout.end is not None else None)

In [ ]:
for context_turn, model_turn in zip(rollout.context_turns, rollout.model_turns):
    for message in context_turn.messages:
        print(f"user  > {message.content[0].text}")
    print(f"model > {_text_of(model_turn)}")

# The policy's per-turn metadata rides on each context turn ...
print(
    "\nparsed guesses:",
    [find_metadata(turn.metadata, ParsedGuess) for turn in rollout.context_turns],
)
# ... and the raw API response rides on each model turn.
last_response = find_metadata(rollout.model_turns[-1].metadata, ModelResponse)
print("last turn usage:", last_response.usage if last_response is not None else None)

## 4. The inline form

For a one-off where the initialization is known at construction,
`context_policy_from_generator` wraps a bare async generator function — no class needed.

In [ ]:
@dataclasses.dataclass(frozen=True)
class FinalAnswer(TaggedDataclass, fxtr_type="org.example.two_questions.FinalAnswer.v1"):
    text: str | None


async def two_questions() -> ContextPolicyGenerator:
    yield ContextTurn(messages=(_user("Name a prime number."),), available_tools=None)
    model_turn = yield ContextTurn(
        messages=(_user("Now double it and reply with just the result."),),
        available_tools=None,
    )
    answer = FinalAnswer(_text_of(model_turn) if model_turn else None)
    yield EndRollout(metadata=(DynamicData.from_tagged(answer),))


inline_rollout = await run_chat_rollout(
    model, context_policy_from_generator(two_questions, initialization=None)
)
print(
    find_metadata(inline_rollout.end.metadata, FinalAnswer)
    if inline_rollout.end is not None
    else None
)

## 5. Rollouts are data

`ChatRollout` is an fxtr entity: storing it and loading it back round-trips the whole record —
turns, status, the model's initialization, and every metadata entry the policy and model
attached, which `find_metadata` decodes back into the types that produced them.

In [ ]:
from fxtr.storage.memory_storage import MemoryStorage

storage = MemoryStorage()
ref = await storage.store(rollout)
restored = await storage.load(ref)
assert restored == rollout
[find_metadata(turn.metadata, ParsedGuess) for turn in restored.context_turns]

## Cleanup

Standalone `ModelAPI`s own their HTTP clients; the caller closes them.

In [ ]:
await api.aclose()